# 2. 短期记忆
LangChain1.x 的短期记忆是三者的组合：
State（会话内部状态） + Checkpointer（持久化机制） + Thread ID（会话作用域）
* State ：默认 存储历史消息列表messages ，通过State 管理历史消息
* Checkpointer ：负责将State 作为检查点持久化保存，检查点是某个时刻的State 快照
* Thread ID ：用于唯一标识State ，LangChain运行时会按照 thread_id 读写State快照

> 这就像玩 RPG 游戏时的“自动存档”：你不需要手动保存，系统在关键节点自动记录，下次进入游戏随时可以从上次的存档点继续。

## 2.1 基于内存的持久化器

这是最便捷的使用方式，适合快速测试或调试。

In [11]:
import os
from dotenv import load_dotenv
from langchain_openai import ChatOpenAI
from langchain.agents import create_agent
from langchain.agents.structured_output import ToolStrategy
from langchain.tools import tool
from pydantic import BaseModel, Field

load_dotenv(override=True, verbose=True, dotenv_path="/Users/skk/Developer/lang-chain-learn/conf/.env")

custom_profile = {
    "max_input_tokens": 128_000
}

model = ChatOpenAI(
    model="gpt-6-sol",
    api_key=os.environ["OPENAI_API_KEY"],
    base_url=os.environ["OPENAI_API_BASE"],
    #profile=custom_profile
)

### 举例1:没有记忆

In [12]:
from langchain.agents import create_agent
from langchain.messages import HumanMessage

agent = create_agent(
    model=model,
    tools=[]
)

print("\n第一轮对话：")
response1 = agent.invoke({
    "messages": [HumanMessage("我叫张三")]
})
print(f"Agent: {response1['messages'][-1].content}")

print("\n第二轮对话：")
response2 = agent.invoke({
    "messages": [HumanMessage("我叫什么？")]
})
print(f"Agent: {response2['messages'][-1].content}")


第一轮对话：
Agent: 你好，张三！很高兴认识你。有什么我可以帮你的吗？

第二轮对话：
Agent: 我还不知道你叫什么。你可以告诉我你的名字吗？


### 举例2:拥有记忆

In [13]:
from langchain.agents import create_agent
from langchain.messages import HumanMessage
from langgraph.checkpoint.memory import InMemorySaver

checkpointer = InMemorySaver()  #1、创建了内存级的记忆存储


agent = create_agent(
    model=model,
    tools=[],
    checkpointer=checkpointer  #2、让agent具备了存储的能力
)

# 3、同一个thread_id共享记忆的。
config = {
    "configurable" : {
        "thread_id" : "1"
    }
}

print("\n第一轮对话：")
response1 = agent.invoke({
    "messages": [HumanMessage("我叫张三")]},
    config=config   # 4、传入invoke()当中
)
print(f"Agent: {response1['messages'][-1].content}")

print("\n第二轮对话：")
response2 = agent.invoke({
    "messages": [HumanMessage("我叫什么？")]},
    config=config
)
print(f"Agent: {response2['messages'][-1].content}")


第一轮对话：
Agent: 你好，张三！很高兴认识你。

第二轮对话：
Agent: 你叫张三。


In [14]:
print("\n第三轮对话：")
response3 = agent.invoke({
    "messages": [HumanMessage("我刚才问了什么问题？")]},
    config=config
)
print(f"Agent: {response3['messages'][-1].content}")


第三轮对话：
Agent: 你刚才问的是：“我叫什么？”


In [15]:
from rich import print as rprint

thread_1_state = agent.get_state(config)
rprint(thread_1_state)

StateSnapshot(
    values={
        'messages': [
            HumanMessage(
                content='我叫张三',
                additional_kwargs={},
                response_metadata={},
                id='e264b7db-9f6a-49ad-bdc0-e2f31f56bc23'
            ),
            AIMessage(
                content='你好，张三！很高兴认识你。',
                additional_kwargs={'refusal': None},
                response_metadata={
                    'token_usage': {
                        'completion_tokens': 15,
                        'prompt_tokens': 10,
                        'total_tokens': 25,
                        'completion_tokens_details': {
                            'accepted_prediction_tokens': None,
                            'audio_tokens': None,
                            'reasoning_tokens': 0,
                            'rejected_prediction_tokens': None,
                            'text_tokens': None
                        },
                        'prompt_tokens_details': {
                            'audio_tokens': None,
                            'cache_write_tokens': 0,
                            'cached_tokens': 0,
                            'image_tokens': None,
                            'text_tokens': None,
                            'cached_creation_tokens': 0
                        }
                    },
                    'model_provider': 'openai',
                    'model_name': 'gpt-6-sol',
                    'system_fingerprint': None,
                    'id': 'resp_0ae395ed262d0e73016ac70c3d48b8819a86ca59ce1d0039c1',
                    'service_tier': 'default',
                    'finish_reason': 'stop',
                    'logprobs': None
                },
                id='lc_run--01a11987-c446-7432-90ab-952db25fd444-0',
                tool_calls=[],
                invalid_tool_calls=[],
                usage_metadata={
                    'input_tokens': 10,
                    'output_tokens': 15,
                    'total_tokens': 25,
                    'input_token_details': {'cache_creation': 0, 'cache_read': 0},
                    'output_token_details': {'reasoning': 0}
                }
            ),
            HumanMessage(
                content='我叫什么？',
                additional_kwargs={},
                response_metadata={},
                id='107492ff-075c-45e1-8378-c1578d4cfb8d'
            ),
            AIMessage(
                content='你叫张三。',
                additional_kwargs={'refusal': None},
                response_metadata={
                    'token_usage': {
                        'completion_tokens': 9,
                        'prompt_tokens': 34,
                        'total_tokens': 43,
                        'completion_tokens_details': {
                            'accepted_prediction_tokens': None,
                            'audio_tokens': None,
                            'reasoning_tokens': 0,
                            'rejected_prediction_tokens': None,
                            'text_tokens': None
                        },
                        'prompt_tokens_details': {
                            'audio_tokens': None,
                            'cache_write_tokens': 0,
                            'cached_tokens': 0,
                            'image_tokens': None,
                            'text_tokens': None,
                            'cached_creation_tokens': 0
                        }
                    },
                    'model_provider': 'openai',
                    'model_name': 'gpt-6-sol',
                    'system_fingerprint': None,
                    'id': 'resp_0b155b1d499d9041016ac70c41836c8199bd6a74f90d050ee0',
                    'service_tier': 'default',
                    'finish_reason': 'stop',
                    'logprobs': None
                },
                id='lc_run--01a11987-d490-7830-a643-ea304b46b0ed-0',
                tool_calls=[],
                invalid_tool_calls=[

### 2.1.3 代码原理解释
关键步骤:
1. 第1步：初始化记忆引擎： checkpointer = InMemorySaver() ——创建一个内存级的记忆存储。
2. 第2步：绑定 Agent：在 create_agent 时传入 checkpointer ，让 Agent 具备状态存储能力
3. 第3步：设定会话 ID：通过 config = {"configurable": {"thread_id": "1"}} 为每次调用指定线程标识。同一个 thread_id 共享记忆，不同 thread_id 完全隔离。
4. 第4步：调用时传入 config：每次 `invoke` 都要带上 `config=config`。创建 Agent 时传了 checkpointer、调用时却没传 `thread_id`，会报错 `ValueError: Checkpointer requires one or more of the following 'configurable' keys: thread_id, checkpoint_ns, checkpoint_id`。

### 2.1.4 第二次调用时发生了什么

举例2 的第二轮只传入了一条新消息“我叫什么？”，模型却答出了“你叫张三”。下图是这次 `invoke` 内部的过程：

![第二次 invoke 时 checkpointer 做了什么](assets/04-invoke-with-checkpointer.svg)

1. **读取**：Agent 开始运行时，先用 `config` 里的 `thread_id` 向 checkpointer 要这个会话最新的检查点，从中恢复出 state。此时 `messages` 里已经有第一轮的 2 条消息。
2. **合并**：把本次传入的新消息合并进 state。`messages` 字段的合并规则是 `add_messages`（第八章 07 笔记 5.3.4）：`id` 与已有消息都不同的消息追加到末尾，`id` 相同的替换原消息。新建的 `HumanMessage` 没有 `id`，所以被追加。
3. **执行**：`model` 节点把这 3 条消息发给大模型，模型的回复同样按 `add_messages` 追加。
4. **保存**：每执行完一步，Agent 就把当时的 state 交给 checkpointer，存成一个新的**检查点**（checkpoint），也就是开头说的“自动存档”。

所以 `response2["messages"]` 有 4 条消息。没有 checkpointer 时（举例1），第 1 步读不到任何东西，state 只能从本次传入的消息开始。

> 这里的“节点”就是第七章 01 笔记 2.4 中 Agent 图里的方框（`model`、`tools` 等），“一步”指执行一轮节点，2.1.6 会结合检查点再解释。

**使用时的两个注意点：**

* **只传新消息，不要再手动拼历史。** 如果像上一节的 `messages1` 那样，把历史消息新建一遍再传进去，这些新建的消息没有 `id`，会被当成新消息再追加一次，历史就重复了。（把上一次返回的 `response["messages"]` 原样传回去倒不会重复：它们带着原来的 `id`，只会替换自己。）
* **系统提示词不在记忆里。** `create_agent(system_prompt=...)` 设置的系统提示词不会存进 state，每次调用模型时才临时加在最前面（第七章 04 笔记 7.1.3）。所以修改系统提示词后，旧会话下一次调用就会用上新的提示词。

### 2.1.5 State 中存了什么

checkpointer 保存的就是 state。前面几章陆续接触过 state，这里系统地整理一遍。举例2 最后一个单元格用 `agent.get_state(config)` 取出了 thread_id 为 `"1"` 的会话当前的状态，输出很长，下面分三层来读：

1. `get_state` 返回的 `StateSnapshot`：state 本身 + 这份存档的档案信息
2. `values`：state 有哪些字段
3. `messages`：每条消息存了什么

#### 1. 读懂 get_state 的输出：StateSnapshot

`get_state` 返回一个 `StateSnapshot`（状态快照）对象，对应这个 thread **最新的检查点**。它有 8 个属性，真正的 state 只是其中的 `values`：

| 属性 | 含义 | 举例2 输出中的值 |
|---|---|---|
| `values` | **state 本身**：各字段当前的值 | `{'messages': [6 条消息]}` |
| `next` | 接下来要执行哪些节点；空元组表示这次运行已经结束 | `()` |
| `config` | 这个检查点的“地址”：`thread_id` + `checkpoint_ns` + `checkpoint_id` | `thread_id='1'`、`checkpoint_ns=''`、`checkpoint_id='1f1c2663-…'` |
| `metadata` | 检查点的附加信息，主要是 `source`（怎么产生的）和 `step`（第几步） | `source='loop'`、`step=7` |
| `created_at` | 检查点的创建时间（UTC 时间） | `'2026-10-07T15:46:00…'` |
| `parent_config` | 上一个检查点的地址 | `checkpoint_id='1f1c2662-…'` |
| `tasks` | 下一步待执行的任务，和 `next` 对应 | `()` |
| `interrupts` | 等待人工处理的中断。第八章人工审批（`HumanInTheLoopMiddleware`）暂停时，这里会有内容 | `()` |

* `checkpoint_id` 每个检查点一个，按时间递增。`checkpoint_ns` 是子图（嵌套在图里的图）用的命名空间，Agent 中始终是空字符串，可以忽略。
* `step=7`：三轮对话一共产生了 9 个检查点（step 从 -1 编号到 7），这是最新的一个，2.1.6 详细解释。
* `metadata` 里的 `ls_integration: 'langchain_create_agent'` 是 LangChain 加的标记，表示这张图由 `create_agent` 创建，了解即可。

#### 2. values：state 有哪些字段

`create_agent` 默认的 state 结构是 `AgentState`（第八章 07 笔记 5.3.3），源码如下（`langchain/agents/middleware/types.py`）：

```python
class AgentState(TypedDict, Generic[ResponseT]):
    messages: Required[Annotated[list[AnyMessage], add_messages]]
    jump_to: NotRequired[Annotated[JumpTo | None, EphemeralValue, PrivateStateAttr]]
    structured_response: NotRequired[Annotated[ResponseT, OmitFromInput]]
```

`Annotated[类型, 说明1, 说明2, ...]` 是在类型后面附加说明。LangGraph 读取这些说明，决定字段怎么合并、要不要保存：

| 字段 | 什么时候有 | 存什么 | 存进检查点吗 |
|---|---|---|---|
| `messages` | 一直有（`Required`） | 对话的全部消息，合并规则是 `add_messages` | **存**，短期记忆的主体 |
| `structured_response` | 设置了 `response_format` 时（第七章 05 笔记） | 结构化输出的结果。`OmitFromInput` 表示它只出现在输出里，不作为输入 | 存 |
| `jump_to` | 中间件要求跳转时（第八章 08 笔记） | 跳转目标，如 `"end"` | **不存**。`EphemeralValue` 表示只在紧接着的下一步有效，之后就清空；`PrivateStateAttr` 表示不出现在输入和返回值里 |
| 自定义字段 | 通过 `state_schema` 添加（自己定义，或中间件自带） | 任何需要跟着会话保存的数据 | 默认存；标注了 `UntrackedValue` 的不存 |

所以举例2 的 `values` 里只有 `messages`：没有设置 `response_format`，`jump_to` 又不会保存。

**中间件自带的字段**也由 checkpointer 保存。最典型的是 `ModelCallLimitMiddleware`（第八章 06 笔记）往 state 里加的两个计数器：

| 字段 | 标注 | 效果 |
|---|---|---|
| `thread_model_call_count` | `PrivateStateAttr` | 随检查点保存，同一个 thread 的多次 `invoke` 会累加。所以 `thread_limit` 必须配合 checkpointer |
| `run_model_call_count` | `UntrackedValue`、`PrivateStateAttr` | 不保存，每次 `invoke` 都从 0 开始。所以 `run_limit` 只管单次运行 |

`TodoListMiddleware`（第八章 05 笔记）的 `todos` 也是这样保存在 state 里的。另外，标注了 `PrivateStateAttr` 的字段不会出现在 `invoke` 的返回值里，要用 `get_state` 才能看到。

**不在 state 里的东西**：系统提示词、工具列表、模型配置都属于 Agent 的配置；`context` 是每次调用时传入的运行时上下文（上一节 1.2.2）。它们都不在 state 里，也不会被 checkpointer 保存。

#### 3. messages：每条消息存了什么

| 消息类型 | 由谁写入 | 保存的主要内容 |
|---|---|---|
| `HumanMessage` | 用户输入，由入口节点 `__start__` 写入 | `content`：用户说的话<br>`id`：传入时没有，由 `add_messages` 自动生成一个 UUID |
| `AIMessage` | `model` 节点 | `content`：回复文本<br>`tool_calls`：要调用的工具及参数，不调用工具时为空列表<br>`id`：以 `lc_run--` 开头<br>`response_metadata`：模型名、结束原因（`finish_reason`）、厂商返回的原始用量等<br>`usage_metadata`：输入、输出的 token 数 |
| `ToolMessage` | `tools` 节点 | `content`：工具的返回值<br>`tool_call_id`：对应哪一次工具调用<br>`name`：工具名<br>`status`：`success` 或 `error` |
| `SystemMessage` | 默认没有 | `system_prompt` 不存进来（见 2.1.4）。只有你自己把系统消息放进 `messages` 传入时才会有 |

结合举例2 的输出看几点：

* **模型返回的元数据也全部保存了。** 每条 `AIMessage` 都带着完整的 `response_metadata` 和 `usage_metadata`，所以输出那么长。记忆里存的是完整的消息对象，不只是文字。
* **每条消息都有 `id`。** `add_messages` 按 `id` 决定追加还是替换；下一节学的删除消息（`RemoveMessage`）也按 `id` 指定删哪一条。
* **`input_tokens` 随轮次增长**（4392 → 4410 → 4433）：每一轮都要把全部历史重新发给模型，对话越长，每次请求越贵。这就是需要裁剪、摘要的原因。（基数有 4000 多，是因为中转服务会额外附加一段系统提示词。）

#### 4. 演示：带工具调用和自定义字段的 state

举例2 没有工具，state 里只有一问一答。下面新建一个 Agent：给它一个查天气的工具，再用 `state_schema` 给 state 加一个自定义字段 `user_name`（第八章 08 笔记 5.3.9），看看 state 里会存下什么。

这个单元格会请求两次模型（一次决定调用工具，一次根据工具结果回答）：

In [16]:
from langchain.agents import AgentState
from langchain.messages import HumanMessage, ToolMessage
from langgraph.checkpoint.memory import InMemorySaver


class MyState(AgentState):
    user_name: str  # 自定义字段：没有合并规则，新值直接覆盖旧值


@tool
def get_weather(city: str) -> str:
    """查询指定城市的天气"""
    return f"{city}：晴，25℃"


demo_agent = create_agent(
    model=model,
    tools=[get_weather],
    state_schema=MyState,
    checkpointer=InMemorySaver(),
)
demo_config = {"configurable": {"thread_id": "demo"}}


def show_state(values):
    """简洁地打印 state：先打印其他字段，再逐条打印消息的关键信息。
    msg.text 只取消息内容中的文字部分，没有文字时为空字符串。"""
    for key, value in values.items():
        if key != "messages":
            print(f"{key} = {value!r}")
    print(f"messages 共 {len(values['messages'])} 条：")
    for i, msg in enumerate(values["messages"]):
        line = f"  [{i}] {type(msg).__name__:<13} text={msg.text!r}"
        if getattr(msg, "tool_calls", None):
            line += f"  tool_calls={[(call['name'], call['args']) for call in msg.tool_calls]}"
        if isinstance(msg, ToolMessage):
            line += f"  tool_call_id={msg.tool_call_id!r}"
        print(line)


# 自定义字段可以和 messages 一起传入
demo_agent.invoke(
    {"messages": [HumanMessage("北京今天天气怎么样？")], "user_name": "张三"},
    demo_config,
)
show_state(demo_agent.get_state(demo_config).values)

user_name = '张三'
messages 共 4 条：
  [0] HumanMessage  text='北京今天天气怎么样？'
  [1] AIMessage     text=''  tool_calls=[('get_weather', {'city': '北京'})]
  [2] ToolMessage   text='北京：晴，25℃'  tool_call_id='call_b61e8f6cd3ec47fb95fd839b275f3c2e'
  [3] AIMessage     text='北京今天晴，气温 25℃。'


* 一次工具调用会在 `messages` 里留下两条消息：带 `tool_calls` 的 `AIMessage`（模型决定调用哪个工具、传什么参数），以及 `ToolMessage`（工具的返回值，用 `tool_call_id` 和前者对应）。它们和问答一样会被保存，下一轮模型能看到“查过天气，结果是什么”。
* 自定义字段 `user_name` 和 `messages` 并列保存在 state 里。

> 这个中转模型返回的 `AIMessage.content` 有时是内容块列表（第四章 01 笔记 1.4.2），调用工具时里面还会有 `function_call` 块，直接打印很乱。所以 `show_state` 打印的是 `msg.text`，只取其中的文字：第 1 条 `AIMessage` 只调用了工具、没有文字，显示为空字符串。

接着在同一个会话里再问一句，这次不传 `user_name`（请求一次模型）：

In [17]:
demo_agent.invoke({"messages": [HumanMessage("我叫什么名字？")]}, demo_config)
show_state(demo_agent.get_state(demo_config).values)

user_name = '张三'
messages 共 6 条：
  [0] HumanMessage  text='北京今天天气怎么样？'
  [1] AIMessage     text=''  tool_calls=[('get_weather', {'city': '北京'})]
  [2] ToolMessage   text='北京：晴，25℃'  tool_call_id='call_b61e8f6cd3ec47fb95fd839b275f3c2e'
  [3] AIMessage     text='北京今天晴，气温 25℃。'
  [4] HumanMessage  text='我叫什么名字？'
  [5] AIMessage     text='你还没告诉我你的名字。'


* `user_name` 还在：这次没有传入，它是从检查点恢复的。自定义字段和消息一样，属于这个 thread 的短期记忆。
* 但模型并不知道用户叫张三：**发给模型的只有 `messages`**，自定义字段只是保存在 state 里。要让模型用上它，需要在中间件里从 `state` 读出来，放进发给模型的请求（第八章 07 笔记），这就是上一节说的上下文工程。

### 2.1.6 InMemorySaver 如何保存记忆

#### 1. 检查点：每一步存一份 state 快照

**检查点**（checkpoint）就是某一时刻 state 的完整快照，再加上一些档案信息：第几步、怎么产生的、上一个检查点是谁。

什么时候存？Agent 是一张由节点组成的图，LangGraph 按“步”（step）执行它：每一步执行一轮节点，节点的返回值合并进 state 后，这一步结束，checkpointer 随即存一个检查点。举例2 的 Agent 没有工具，一次 `invoke` 会存 3 个检查点：

| 检查点 | `source` | 时机 | `next` |
|---|---|---|---|
| 第 1 个 | `input` | 刚收到输入，还没执行任何节点。新消息暂存在入口节点 `__start__` 那里，还没进 `messages` | `('__start__',)` |
| 第 2 个 | `loop` | `__start__` 节点执行完，新消息已合并进 `messages` | `('model',)` |
| 第 3 个 | `loop` | `model` 节点执行完，模型的回复已追加 | `()`，运行结束 |

`agent.get_state_history(config)` 按从新到旧的顺序列出一个 thread 的全部检查点。下面列出举例2 的会话（不请求模型）：

In [22]:
for snapshot in agent.get_state_history(config):
    print(
        f"step={snapshot.metadata['step']:>2}  "
        f"source={snapshot.metadata['source']:<5}  "
        f"messages {len(snapshot.values['messages'])} 条  "
        f"next={snapshot.next}  "
        f"最后一条消息：{snapshot.values['messages'][-1].content if snapshot.values['messages'] else '无'}"
    )

step= 7  source=loop   messages 6 条  next=()  最后一条消息：你刚才问的是：“我叫什么？”
step= 6  source=loop   messages 5 条  next=('model',)  最后一条消息：我刚才问了什么问题？
step= 5  source=input  messages 4 条  next=('__start__',)  最后一条消息：你叫张三。
step= 4  source=loop   messages 4 条  next=()  最后一条消息：你叫张三。
step= 3  source=loop   messages 3 条  next=('model',)  最后一条消息：我叫什么？
step= 2  source=input  messages 2 条  next=('__start__',)  最后一条消息：你好，张三！很高兴认识你。
step= 1  source=loop   messages 2 条  next=()  最后一条消息：你好，张三！很高兴认识你。
step= 0  source=loop   messages 1 条  next=('model',)  最后一条消息：我叫张三
step=-1  source=input  messages 0 条  next=('__start__',)  最后一条消息：无


* 三轮对话共 9 个检查点，`step` 从 -1 编号到 7，跨轮次连续计数。`get_state` 返回的就是最上面 step 7 那个。
* 每个检查点的 `parent_config` 指向上一个，9 个检查点连成一条链。
* `source=input` 的检查点里，`messages` 还不包含本轮的新消息，所以条数和上一轮结束时一样。

有工具调用时，每多执行一个节点就多一个检查点。看看刚才的演示会话（不请求模型）：

In [19]:
for snapshot in demo_agent.get_state_history(demo_config):
    types = [type(m).__name__.removesuffix("Message") for m in snapshot.values["messages"]]
    print(
        f"step={snapshot.metadata['step']:>2}  "
        f"source={snapshot.metadata['source']:<5}  "
        f"next={str(snapshot.next):<14}  "
        f"messages={types}"
    )

step= 6  source=loop   next=()              messages=['Human', 'AI', 'Tool', 'AI', 'Human', 'AI']
step= 5  source=loop   next=('model',)      messages=['Human', 'AI', 'Tool', 'AI', 'Human']
step= 4  source=input  next=('__start__',)  messages=['Human', 'AI', 'Tool', 'AI']
step= 3  source=loop   next=()              messages=['Human', 'AI', 'Tool', 'AI']
step= 2  source=loop   next=('model',)      messages=['Human', 'AI', 'Tool']
step= 1  source=loop   next=('tools',)      messages=['Human', 'AI']
step= 0  source=loop   next=('model',)      messages=['Human']
step=-1  source=input  next=('__start__',)  messages=[]


第一轮依次执行 `model → tools → model` 三个节点，加上 `input` 和 `__start__` 两个检查点，共 5 个；第二轮没有调用工具，是 3 个。

> 保存每一步的快照，不只是为了记住对话。第八章的人工审批就靠它实现：运行在执行工具前暂停时，当时的 state 已经存进检查点；人做出决定后，Agent 从这个检查点继续执行。LangGraph 还可以从历史上的任意一个检查点重新运行（称为“时间旅行”），了解即可。

#### 2. InMemorySaver 内部：三个字典

`InMemorySaver` 的源码只有几百行（`langgraph/checkpoint/memory/__init__.py`），数据全部放在三个 Python 字典里：

| 字典 | 键 | 值 |
|---|---|---|
| `storage` | `thread_id` → `checkpoint_ns` → `checkpoint_id`，三层嵌套 | 三元组：检查点主体（**不含** state 的值，只记各字段的版本号）、元数据、父检查点的 id |
| `blobs` | `(thread_id, checkpoint_ns, 通道名, 版本号)` | 某个字段在某个版本的值 |
| `writes` | `(thread_id, checkpoint_ns, checkpoint_id)` | 从这个检查点出发执行下一步时，各节点写出的更新。这一步中途出错时，已完成节点的结果不会丢，恢复时不用重跑 |

这里的**通道**（channel）可以先理解成 state 的字段：LangGraph 为 state 的每个字段建一个通道，字段名就是通道名。此外还有几个 LangGraph 自己用的内部通道（`__start__`、`branch:to:model` 等），记录的是输入和“下一步该执行哪个节点”，不属于 state，`get_state` 的 `values` 里也看不到。

为什么 state 的值要单独放在 `blobs` 里？因为每一步通常只有少数字段变化。每个通道有一个版本号，被更新一次就加 1；检查点只记录“此刻每个通道是哪个版本”，没变化的通道不用重复保存。

![InMemorySaver 内部的三个字典](assets/05-inmemorysaver-internals.svg)

下面直接查看举例2 的 `checkpointer` 里的数据（不请求模型）。这些是内部属性，只用于学习，实际项目中用 `get_state`、`get_state_history` 读取：

In [20]:
# 1. storage：每个 thread 存了多少个检查点
for thread_id, ns_dict in checkpointer.storage.items():
    for checkpoint_ns, checkpoints in ns_dict.items():
        print(f"storage：thread_id={thread_id!r}  checkpoint_ns={checkpoint_ns!r}  共 {len(checkpoints)} 个检查点")

# 2. 最新检查点的主体：只有各通道的版本号
#    （get_tuple 返回前会按版本号从 blobs 取值，补上 channel_values，这里先跳过它）
checkpoint = checkpointer.get_tuple(config).checkpoint
print("\n检查点主体的键：", [key for key in checkpoint if key != "channel_values"])
print("各通道的版本号：", {name: int(v.split(".")[0]) for name, v in checkpoint["channel_versions"].items()})

# 3. blobs：messages 通道每个版本存了几条消息
print()
for (thread_id, checkpoint_ns, channel, version), blob in checkpointer.blobs.items():
    if channel == "messages":
        messages = checkpointer.serde.loads_typed(blob)  # 存的是序列化后的字节，先还原成消息对象
        print(f"blobs：messages 版本 {int(version.split('.')[0])} → {len(messages)} 条消息")

storage：thread_id='1'  checkpoint_ns=''  共 9 个检查点

检查点主体的键： ['v', 'ts', 'id', 'channel_versions', 'versions_seen', 'updated_channels']
各通道的版本号： {'__start__': 8, 'messages': 9, 'branch:to:model': 9}

blobs：messages 版本 2 → 1 条消息
blobs：messages 版本 3 → 2 条消息
blobs：messages 版本 5 → 3 条消息
blobs：messages 版本 6 → 4 条消息
blobs：messages 版本 8 → 5 条消息
blobs：messages 版本 9 → 6 条消息


* `messages` 每更新一次，`blobs` 里就多存一份**完整的**消息列表，而不是只存新增的那条。6 条消息一共存了 1+2+3+4+5+6 = 21 份副本，对话越长，占用增长越快。
* 值以序列化后的字节保存（默认由 `JsonPlusSerializer` 转成 msgpack 格式），即使在内存里也是一份独立的副本：修改 `get_state` 返回的消息对象，不会改动 checkpointer 里存的记忆。

> 一个容易遇到的提示：如果 state 里存了自己定义的类（比如 `response_format` 用的、定义在 notebook 里的 Pydantic 类），读取检查点时 LangGraph 会提示 `Deserializing unregistered type __main__.xxx from checkpoint...`，意思是这个类没有登记为可信类型，以后的版本可能拒绝还原。学习时可以忽略。

#### 3. 读和写的过程（源码简化）

**写入 `put`**：每一步结束时调用。

```python
def put(self, config, checkpoint, metadata, new_versions):
    thread_id = config["configurable"]["thread_id"]
    checkpoint_ns = config["configurable"]["checkpoint_ns"]
    values = checkpoint.pop("channel_values")         # 把 state 的值从检查点里拿出来
    for channel, version in new_versions.items():     # 只处理这一步更新过的通道
        self.blobs[(thread_id, checkpoint_ns, channel, version)] = 序列化(values[channel])
    self.storage[thread_id][checkpoint_ns][checkpoint["id"]] = (
        序列化(checkpoint),                            # 主体：只剩版本号等信息
        序列化(metadata),
        config["configurable"].get("checkpoint_id"),   # 父检查点：上一步的 id
    )
```

**读取 `get_tuple`**：每次 `invoke` 开始时调用，`get_state` 也是调用它。

```python
def get_tuple(self, config):
    thread_id = config["configurable"]["thread_id"]
    checkpoint_ns = config["configurable"].get("checkpoint_ns", "")
    checkpoints = self.storage[thread_id][checkpoint_ns]
    if not checkpoints:
        return None                                   # 新 thread：没有记忆，从空 state 开始
    checkpoint_id = max(checkpoints.keys())           # id 按时间递增，最大的就是最新的
    checkpoint, metadata, parent_id = checkpoints[checkpoint_id]
    checkpoint = 反序列化(checkpoint)
    checkpoint["channel_values"] = {                  # 按版本号到 blobs 里取值，拼回 state
        channel: 反序列化(self.blobs[(thread_id, checkpoint_ns, channel, version)])
        for channel, version in checkpoint["channel_versions"].items()
    }
    return CheckpointTuple(config=..., checkpoint=checkpoint, metadata=反序列化(metadata), ...)
```

* thread 之间的隔离就来自这里：所有键都以 `thread_id` 开头，换一个 `thread_id`，`self.storage[thread_id]` 就是空字典，读不到任何检查点。
* 如果 `config` 里还带了 `checkpoint_id`，`get_tuple` 会取那个指定的检查点，而不是最新的。这就是从历史检查点继续运行的入口。
* 真实源码还处理了 `writes`、空值等细节，主要逻辑和上面一致。

#### 4. InMemorySaver 的特点与局限

| 特点 | 说明 | 影响 |
|---|---|---|
| 数据在当前 Python 进程的内存里 | 重启 kernel、程序退出，记忆全部消失；多个进程（如部署了多个服务实例）之间也不共享 | 只适合学习和测试 |
| 每一步都存完整快照 | 如上所述，`messages` 的每个版本都是完整的列表 | 长对话占用内存多；对话本身也越来越长，需要配合下一节的裁剪、删除、摘要 |
| 旧检查点不会自动删除 | 举例2 的 9 个检查点一直都在 | 可以用 `checkpointer.delete_thread("1")` 删除某个会话的全部检查点、值和更新 |
| 接口和其他 checkpointer 相同 | `put`、`get_tuple`、`list` 等是所有 checkpointer 的统一接口 | 换成 `SqliteSaver`、`PostgresSaver` 等，只需改创建 checkpointer 的那一行，Agent 的代码不用动 |

> 老教程中常见的 `MemorySaver` 是 `InMemorySaver` 的旧名字（源码中 `MemorySaver = InMemorySaver`），两者完全一样。

### 2.1.7 小结

**1. InMemorySaver 如何实现记忆存储？**

* 每次 `invoke` 开始时，用 `thread_id` 读出这个会话最新的检查点，恢复 state；运行中每执行完一步，就把当时的 state 存成一个新检查点。
* 数据放在三个字典里：`storage` 存检查点主体、元数据和父检查点 id；`blobs` 按“通道 + 版本号”存 state 各字段的值；`writes` 存每一步各节点写出的更新。所有键都以 `thread_id` 开头，所以不同会话互不干扰。

**2. 它存储了什么？——state 的快照**

| 层次 | 内容 |
|---|---|
| state 的字段 | `messages`（一直有）、`structured_response`（用了 `response_format` 时）、自定义字段和中间件的字段；`jump_to` 和标注了 `UntrackedValue` 的字段不保存 |
| messages 中的消息 | `HumanMessage`、`AIMessage`（含 `tool_calls`、token 用量等元数据）、`ToolMessage`，每条都有 `id` |
| 检查点的档案信息 | `step`、`source`、创建时间、父检查点、下一步要执行的节点 |
| 不保存的 | 系统提示词、工具、模型配置、`context` |